# Decorators

Decorators allow you to modify or enhance the behavior of functions or methods without permanently modifying their source code. They are heavily used in Python frameworks like Flask and Django.

Before understanding decorators, we first need to understand that **functions are first-class objects** in Python. This means they can be assigned to variables, passed as arguments, and returned from other functions.

## 1. Functions inside Functions

Let's look at how we can define a function inside another function and return it.

In [1]:
def parent_function():
    print("Printing from the parent function")

    def child_function():
        print("Printing from the child function")

    return child_function

# Save the returned child_function to a variable
my_func = parent_function()

# Execute it
my_func()

Printing from the parent function
Printing from the child function


## 2. Creating a Simple Decorator

A decorator is simply a function that takes another function as an argument, adds some functionality, and returns a new function. Python uses the `@` symbol as syntactic sugar for this process.

In [2]:
def my_decorator(func):
    def wrapper():
        print("Something is happening BEFORE the function is called.")
        func()
        print("Something is happening AFTER the function is called.")
    return wrapper

# Using the @ syntax automatically passes say_hello into my_decorator
@my_decorator
def say_hello():
    print("Hello! The actual function is running.")

say_hello()

Something is happening BEFORE the function is called.
Hello! The actual function is running.
Something is happening AFTER the function is called.


## 3. Decorating Functions with Arguments

If the function you are decorating takes arguments, the inner `wrapper` function needs to accept them. We use `*args` and `**kwargs` so the decorator works for functions with any number of arguments.

In [3]:
def uppercase_decorator(func):
    def wrapper(*args, **kwargs):
        # Execute the original function and capture its result
        result = func(*args, **kwargs)
        # Modify the result
        return result.upper()
    return wrapper

@uppercase_decorator
def greet(name):
    return f"Good morning, {name}!"

print(greet("Alice"))

GOOD MORNING, ALICE!


## 4. Preserving Function Identity (`functools.wraps`)

A side effect of decorators is that they hide the original function's name and docstring (because it gets replaced by the `wrapper` function). To fix this, we use the `@wraps` decorator from the `functools` module.

In [4]:
import functools

def proper_decorator(func):
    @functools.wraps(func)  # Preserves identity of 'func'
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@proper_decorator
def do_nothing():
    """This function literally does nothing."""
    pass

# Because we used @wraps, this will correctly print 'do_nothing' instead of 'wrapper'
print("Function name:", do_nothing.__name__)
print("Function docstring:", do_nothing.__doc__)

Function name: do_nothing
Function docstring: This function literally does nothing.


## 5. Real-World Example: Execution Timer

Let's build a practical decorator that measures how long a function takes to run.

In [5]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper_timer(*args, **kwargs):
        start_time = time.perf_counter()    # 1. Record start time
        value = func(*args, **kwargs)       # 2. Execute the function
        end_time = time.perf_counter()      # 3. Record end time

        run_time = end_time - start_time
        print(f"Finished {func.__name__!r} in {run_time:.4f} secs")
        return value
    return wrapper_timer

@timer
def waste_some_time(num_times):
    for _ in range(num_times):
        sum([i**2 for i in range(1000)])

waste_some_time(500)

Finished 'waste_some_time' in 0.0436 secs
